# 💣 DM2026 AI Game Jam Golden Demo — 離散數學《踩地雷》人機協作範例

> **課程**：115學年第1學期《離散數學》(VNU 234) — `Grace-TA/DM2026`  
> **目的**：提供全班同學一份 **Google Colab 與 GitHub Codespaces 雙棲可運行** 的 A+ 滿分示範筆記本，完整示範如何在 20–25 分鐘發表與 3–5 分鐘 Demo Video 中呈現 **5 大離散數學觀念逆向工程**、**V1→V3 Prompt 進化史** 與 **100× 規模壓力測試**。

In [ ]:
# Cell 1: Standalone Setup (Works directly in Google Colab OR GitHub Codespaces)
import math, random, time
from collections import deque
from enum import Enum
import matplotlib.pyplot as plt
import numpy as np

print('✅ Discrete Math Minesweeper Golden Demo Environment Ready!')

## Part 1：V1 AI 壞程式碼逆向工程（Why Raw LLM Prompts Fail Math Defense）
以下示範未約束 AI 初版常見的四大致命離散結構瑕疵：`[[0]*W]*H` 淺拷貝、$O(N^2)$ 列表查雷、無保護遞迴 `RecursionError` 與缺少 FSM 吸收態。

In [ ]:
class MinesweeperV1Buggy:
    def __init__(self, rows=9, cols=9, num_mines=10, seed=42):
        self.rows, self.cols, self.num_mines = rows, cols, num_mines
        self.rng = random.Random(seed)
        self.logic_board = [[0] * cols] * rows  # BUG 1: Shallow Copy Aliasing!
        self.mines_list = []
        while len(self.mines_list) < num_mines:
            cell = (self.rng.randint(0, rows-1), self.rng.randint(0, cols-1))
            if cell not in self.mines_list:    # BUG 2: O(|M|) linear scan
                self.mines_list.append(cell)

    def reveal_recursive_unsafe(self, r, c, visited=None):
        if visited is None: visited = set()
        if not (0 <= r < self.rows and 0 <= c < self.cols) or (r, c) in visited: return 0
        visited.add((r, c))
        opened = 1
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr or dc: opened += self.reveal_recursive_unsafe(r+dr, c+dc, visited)
        return opened

v1 = MinesweeperV1Buggy(9, 9, 10, seed=42)
v1.logic_board[0][0] = 99
print('BUG 1 實測 - 修改 (0,0) 後，第 5 列也變成:', v1.logic_board[5][0])

## Part 2：V3 Hero 引擎 — 五大離散數學觀念完整實作 (`Lab 1`–`Lab 9`)

In [ ]:
class GameState(Enum):
    INIT = 'INIT'
    PLAYING = 'PLAYING'
    WON = 'WON'
    LOST = 'LOST'

class MinesweeperHero:
    DIRS_8 = ((-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1))

    def __init__(self, rows=9, cols=9, num_mines=10, seed=42):
        self.rows, self.cols, self.num_mines = rows, cols, num_mines
        self.rng = random.Random(seed)
        self.state = GameState.INIT
        # 1. Lab1-2: Cartesian Grid + Row-Isolated List Comprehension
        self.logic_board = [[0 for _ in range(cols)] for _ in range(rows)]
        self.revealed, self.flagged = set(), set()
        # 2. Lab3 & Lab5: Set-based collision-free sample -> |M| == num_mines
        all_cells = [(r, c) for r in range(rows) for c in range(cols)]
        self.mines = set(self.rng.sample(all_cells, num_mines))
        for r, c in self.mines: self.logic_board[r][c] = -1
        for r in range(rows):
            for c in range(cols):
                if (r, c) not in self.mines:
                    self.logic_board[r][c] = sum(1 for nr, nc in self.get_neighbors(r, c) if (nr, nc) in self.mines)
        self.state = GameState.PLAYING

    def is_valid(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols

    def get_neighbors(self, r, c):
        # 3. Lab4: 8-Neighbor Binary Relation with strict closed-interval guard
        return [(r+dr, c+dc) for dr, dc in self.DIRS_8 if self.is_valid(r+dr, c+dc)]

    def reveal(self, r, c):
        # 5. Lab9: FSM Absorbing State Guard
        if self.state != GameState.PLAYING or not self.is_valid(r, c): return self.state, 0
        if (r, c) in self.mines:
            self.state = GameState.LOST
            return self.state, 0
        # 4. Lab6-8: Iterative BFS (deque) with Mark-on-Enqueue -> O(V+E)
        q = deque([(r, c)])
        seen = {(r, c)}
        opened = 0
        while q:
            cr, cc = q.popleft()
            if (cr, cc) in self.revealed or (cr, cc) in self.mines: continue
            self.revealed.add((cr, cc))
            opened += 1
            if self.logic_board[cr][cc] == 0:
                for nr, nc in self.get_neighbors(cr, cc):
                    if (nr, nc) not in self.revealed and (nr, nc) not in seen:
                        seen.add((nr, nc))
                        q.append((nr, nc))
        if len(self.revealed) == self.rows * self.cols - self.num_mines:
            self.state = GameState.WON
        return self.state, opened

hero = MinesweeperHero(9, 9, 10, seed=42)
z = next((r, c) for r in range(9) for c in range(9) if hero.logic_board[r][c] == 0)
st, op = hero.reveal(*z)
print(f'First zero cell {z} -> BFS opened {op} cells | State={st.value} | C(81,10)={math.comb(81,10):,}')

## Part 3：100× 規模放大壓測（Math Defense What-if 實證）

In [ ]:
v1_big = MinesweeperV1Buggy(100, 100, 0, seed=42)
try:
    v1_big.reveal_recursive_unsafe(0, 0)
except RecursionError:
    print('❌ V1 100x100 (10,000 cells) crashed with RecursionError!')

t0 = time.perf_counter()
v3_big = MinesweeperHero(100, 100, 0, seed=42)
st_big, op_big = v3_big.reveal(0, 0)
ms_big = (time.perf_counter() - t0) * 1000
print(f'✅ V3 100x100 (10,000 cells) survived: state={st_big.value}, opened={op_big}, time={ms_big:.2f}ms')
assert st_big == GameState.WON and op_big == 10000